# Fase 3.9 - pilotos combinados Open Images + AIGI

Esta etapa verifica viabilidade. **Nenhuma celula inicia treinamento.**
O piloto baixa arquivos completos para /content, consulta o manifesto anterior
somente para excluir UIDs usados e nao le o Mirage nem as particoes de teste.
O Open Images verifica dez fotografias de autores distintos. O AIGI verifica
novos UIDs nos grupos FLUX.1_dev, PA_alpha e SD_2.1 e testa cinco URLs reais
de cada grupo. Os numeros finais da 3.9 serao fixados depois destes resultados.


In [1]:
# 1. Dependencias para os pilotos. GPU nao e necessaria nesta etapa.
import sys, subprocess
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       'cryptography', 'huggingface_hub', 'pyarrow', 'boto3', 'Pillow'])
print('Dependencias disponiveis.', flush=True)


Dependencias disponiveis.


In [2]:
# 2. Monta o Drive; os pilotos leem somente o manifesto AIGI anterior.
from google.colab import drive
from pathlib import Path
import shutil
drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/TCC_Final')
if not ROOT.is_dir():
    raise FileNotFoundError(ROOT)
print('Drive:', ROOT, '| livre:', round(shutil.disk_usage(ROOT).free / 1e9, 2), 'GB')
print('/content livre:', round(shutil.disk_usage('/content').free / 1e9, 2), 'GB')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive: /content/drive/MyDrive/TCC_Final | livre: 4.89 GB
/content livre: 247.12 GB


In [3]:
# 3. Desbloqueio seguro da credencial GitHub existente (formatos antigo e novo).
import base64, getpass, json
from cryptography.fernet import Fernet, InvalidToken
from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.primitives.kdf.pbkdf2 import PBKDF2HMAC
credential_path = ROOT / 'credenciais/github.enc'
credential = json.loads(credential_path.read_text(encoding='utf-8'))
if not isinstance(credential, dict) or 'salt' not in credential or ('token' in credential) == ('ciphertext' in credential):
    raise ValueError('Formato da credencial GitHub não reconhecido')
if 'ciphertext' in credential:
    if credential.get('version') != 1: raise ValueError('Versão de credencial não suportada')
    iterations, encrypted = int(credential['iterations']), credential['ciphertext']
else:
    iterations, encrypted = 600000, credential['token']
if iterations < 600000: raise ValueError('KDF insuficiente')
password = getpass.getpass('Senha de desbloqueio do GitHub: ')
key = base64.urlsafe_b64encode(PBKDF2HMAC(algorithm=hashes.SHA256(), length=32, salt=base64.b64decode(credential['salt']), iterations=iterations).derive(password.encode()))
try:
    token = Fernet(key).decrypt(encrypted.encode()).decode()
except InvalidToken:
    raise ValueError('Senha incorreta ou credencial corrompida') from None
finally:
    del password, key
print('Credencial desbloqueada nesta sessão.')

Credencial desbloqueada nesta sessão.


In [4]:
# 4. Snapshot privado imutável da branch, somente leitura; sem token em URL ou Git.
import io, re, shutil, tempfile, time, urllib.request, urllib.error, zipfile
API = 'https://api.github.com/repos/vicemnovo/TCC_Final'
BRANCH = 'fase3-ajuste-fino'
def github_read(url):
    req = urllib.request.Request(url, headers={'Authorization': f'Bearer {token}', 'Accept': 'application/vnd.github+json', 'User-Agent': 'TCC-Fase38-Colab'})
    try:
        with urllib.request.urlopen(req, timeout=120) as response:
            chunks, size, notice = [], 0, time.monotonic()
            while block := response.read(1024 * 1024):
                chunks.append(block); size += len(block)
                if time.monotonic() - notice >= 30:
                    print(f'Download do código: {size / 1048576:.1f} MiB', flush=True); notice = time.monotonic()
            return b''.join(chunks)
    except urllib.error.HTTPError as error:
        raise RuntimeError(f'GitHub HTTP {error.code}; confira token, repositório e branch') from None
try:
    commit = json.loads(github_read(f'{API}/branches/{BRANCH}'))['commit']['sha']
    assert re.fullmatch(r'[0-9a-f]{40}', commit)
    CODE_ROOT = Path('/content/tcc_codigo') / commit
    required = ('experimentos/fase3_9_piloto_aigi.py', 'experimentos/fase3_7_openimages_pilot.py', 'experimentos/fase3_8_aigi_dados.py', 'experimentos/fase3_8_aigi_piloto.py')
    if not CODE_ROOT.exists():
        print('Baixando snapshot do GitHub...', flush=True)
        archive = github_read(f'{API}/zipball/{commit}')
        CODE_ROOT.parent.mkdir(parents=True, exist_ok=True)
        with tempfile.TemporaryDirectory(prefix='tcc37_', dir=CODE_ROOT.parent) as temporary:
            staged = Path(temporary)
            with zipfile.ZipFile(io.BytesIO(archive)) as bundle:
                for item in bundle.infolist():
                    parts = Path(item.filename).parts
                    if item.is_dir() or len(parts) < 2: continue
                    relative = Path(*parts[1:])
                    if relative.is_absolute() or '..' in relative.parts: raise ValueError('Snapshot contém caminho inseguro')
                    target = staged / relative
                    target.parent.mkdir(parents=True, exist_ok=True)
                    with bundle.open(item) as source, target.open('wb') as output: shutil.copyfileobj(source, output)
            if not all((staged / name).is_file() for name in required):
                raise FileNotFoundError('Publique primeiro os arquivos do piloto 3.9 na branch')
            staged.rename(CODE_ROOT)
    assert all((CODE_ROOT / name).is_file() for name in required)
finally:
    del token
print('Código sincronizado:', commit)

Baixando snapshot do GitHub...
Código sincronizado: a8c0a1c09e25a53142198c8f2ac7641a89454e37


In [5]:
# 5. Comandos e progresso periódico; nenhum treino é iniciado aqui.
import os, queue, subprocess, sys, threading, time
OPEN = CODE_ROOT / 'experimentos/fase3_7_openimages_pilot.py'
AIGI = CODE_ROOT / 'experimentos/fase3_9_piloto_aigi.py'
for path in (OPEN, AIGI):
    if not path.is_file(): raise FileNotFoundError(path)
ENV = os.environ.copy()
ENV['TCC_CODE_ROOT'] = str(CODE_ROOT)
def run_live(label, command, *, child_env=None):
    print(label + ': iniciando', flush=True)
    process = subprocess.Popen(command, env=ENV if child_env is None else child_env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    lines = queue.Queue()
    def reader():
        for line in process.stdout: lines.put(line)
        lines.put(None)
    threading.Thread(target=reader, daemon=True).start()
    begin = last = time.monotonic()
    while True:
        try:
            line = lines.get(timeout=5)
            if line is None: break
            print(line, end='', flush=True)
            last = time.monotonic()
        except queue.Empty:
            if time.monotonic() - last >= 30:
                print(f'{label}: ativo há {(time.monotonic()-begin)/60:.1f} min; sem nova saída', flush=True)
                last = time.monotonic()
    code = process.wait()
    if code: raise RuntimeError(f'{label} falhou (código {code}); veja o log acima')
    print(f'{label}: concluído em {(time.monotonic()-begin)/60:.1f} min', flush=True)
print('Código:', CODE_ROOT)

Código: /content/tcc_codigo/a8c0a1c09e25a53142198c8f2ac7641a89454e37


In [6]:
# 6. PILOTO OPEN IMAGES: CSV inteiro + dez JPEGs de autores distintos em /content.
# Nao salva divisao de treino nem checkpoint. Pode demorar no primeiro download.
run_live('piloto Open Images 3.9',
         [sys.executable, '-u', str(OPEN), '--samples', '10'])


piloto Open Images 3.9: iniciando
CSV completo: 638.4 MB | SHA-256=05f3d68dbbb03728d1a37e51479f4f35c062b871e1a6cae8c4cefbe0e0c80ed0
Metadados: 250,000 linhas | elegíveis=249,515 | autores=36,927
Metadados: 500,000 linhas | elegíveis=499,052 | autores=49,423
Metadados: 750,000 linhas | elegíveis=748,542 | autores=57,769
Metadados: 1,000,000 linhas | elegíveis=998,086 | autores=64,057
Metadados: 1,250,000 linhas | elegíveis=1,247,618 | autores=69,153
Metadados: 1,500,000 linhas | elegíveis=1,497,142 | autores=73,427
Piloto: 1/10 JPEGs legíveis
Piloto: 2/10 JPEGs legíveis
Piloto: 3/10 JPEGs legíveis
Piloto: 4/10 JPEGs legíveis
Piloto: 5/10 JPEGs legíveis
Piloto: 6/10 JPEGs legíveis
Piloto: 7/10 JPEGs legíveis
Piloto: 8/10 JPEGs legíveis
Piloto: 9/10 JPEGs legíveis
Piloto: 10/10 JPEGs legíveis
{
  "metadata_rows": 1743042,
  "eligible_images": 1739728,
  "distinct_authors": 76961,
  "pilot_rows": [
    {
      "image_id": "6f30d4e12459ab40",
      "author_group": "flickr:gossamandan",
    

In [7]:
# 7. PILOTO AIGI: confere UIDs novos e cinco fotos reais por gerador.
import base64, getpass, json
from cryptography.fernet import Fernet, InvalidToken
from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.primitives.kdf.pbkdf2 import PBKDF2HMAC
hf_env = ENV.copy()
hf_path = ROOT / 'credenciais/huggingface.enc'
if hf_path.is_file():
    credential = json.loads(hf_path.read_text(encoding='utf-8'))
    if credential.get('version') != 1 or not {'salt', 'iterations', 'ciphertext'} <= credential.keys():
        raise ValueError('Formato da credencial Hugging Face nao reconhecido')
    if int(credential['iterations']) < 600000:
        raise ValueError('Credencial Hugging Face com KDF insuficiente')
    password = getpass.getpass('Senha de desbloqueio do Hugging Face: ')
    key = base64.urlsafe_b64encode(PBKDF2HMAC(
        algorithm=hashes.SHA256(), length=32,
        salt=base64.b64decode(credential['salt']),
        iterations=int(credential['iterations'])).derive(password.encode()))
    try:
        secret = Fernet(key).decrypt(credential['ciphertext'].encode()).decode()
    except InvalidToken:
        raise ValueError('Senha incorreta ou credencial Hugging Face corrompida') from None
    finally:
        del password, key
    hf_env['HF_TOKEN'] = secret
    del secret
    print('Credencial Hugging Face desbloqueada para esta preparacao.', flush=True)
else:
    print('Credencial Hugging Face cifrada ausente; AIGI publico sera acessado sem autenticacao.', flush=True)
try:
    run_live('piloto AIGI 3.9',
             [sys.executable, '-u', str(AIGI), '--root', str(ROOT)],
             child_env=hf_env)
finally:
    hf_env.pop('HF_TOKEN', None)


Credencial Hugging Face desbloqueada para esta preparacao.
piloto AIGI 3.9: iniciando
Baixando/verificando arquivo completo: data/fake-00000-of-00015.parquet
  pronto: 495.2 MB
Baixando/verificando arquivo completo: data/fake-00005-of-00015.parquet
  pronto: 517.3 MB
Baixando/verificando arquivo completo: data/fake-00010-of-00015.parquet
  pronto: 413.8 MB
Baixando/verificando arquivo completo: data/fake-00011-of-00015.parquet
  pronto: 400.6 MB
Baixando/verificando arquivo completo: real_images_metadata.csv
  pronto: 1.8 MB
FLUX.1_dev: 129 UIDs novos pareáveis em 1 shard(s); 1600 UIDs examinados
PA_alpha: 129 UIDs novos pareáveis em 1 shard(s); 1600 UIDs examinados
SD_2.1: 311 UIDs novos pareáveis em 2 shard(s); 3200 UIDs examinados
FLUX.1_dev: foto real 1/5 legível; lado curto=156
FLUX.1_dev: foto real 2/5 indisponível (HTTPError)
FLUX.1_dev: foto real 3/5 legível; lado curto=160
FLUX.1_dev: foto real 4/5 indisponível (UnidentifiedImageError)
FLUX.1_dev: foto real 5/5 indisponível (H

In [8]:
# 8. Resumos dos pilotos para decidir o tamanho e as divisoes da 3.9.
for label, path in (
    ('Open Images', Path('/content/fase3_7_openimages_pilot/pilot_summary.json')),
    ('AIGI', Path('/content/fase3_9_aigi_piloto/pilot_summary.json')),
):
    if not path.is_file():
        print(label, ': piloto ainda nao concluido')
        continue
    result = json.loads(path.read_text(encoding='utf-8'))
    if label == 'Open Images':
        print(label, '| elegiveis=', result['eligible_images'],
              '| autores=', result['distinct_authors'],
              '| JPEGs verificados=', len(result['sample_images']))
    else:
        print(label, '| grupos=', json.dumps(result['groups'], ensure_ascii=False))
print('Envie esta saida e eventuais erros dos pilotos. Nenhum treino foi iniciado.')


Open Images | elegiveis= 1739728 | autores= 76961 | JPEGs verificados= 10
AIGI | grupos= {"FLUX.1_dev": {"new_paired_https_candidates": 129, "probes_attempted": 5, "probes_usable": 0, "probe_size": {"under_256": 2}, "probe_sources": {}, "probe_failures": {"HTTPError": 2, "UnidentifiedImageError": 1}}, "PA_alpha": {"new_paired_https_candidates": 129, "probes_attempted": 5, "probes_usable": 0, "probe_size": {"under_256": 3}, "probe_sources": {}, "probe_failures": {"gaierror": 1, "URLError": 1}}, "SD_2.1": {"new_paired_https_candidates": 311, "probes_attempted": 5, "probes_usable": 2, "probe_size": {"under_256": 1, "256_plus": 2}, "probe_sources": {"Laion-2B-en-aesthetic": 1, "MSCOCO": 1}, "probe_failures": {"HTTPError": 2}}}
Envie esta saida e eventuais erros dos pilotos. Nenhum treino foi iniciado.
